# Quant Stack — Full Research Workflow Template

This notebook demonstrates the complete pipeline from raw data through to a
performance tear sheet, using the UK large-cap universe defined in
`config/settings.yaml`.

**Pipeline stages:**
1. Load / generate data
2. Feature engineering via `FeaturePipeline`
3. Quick ML model comparison (AutoML screening)
4. Walk-forward cross-validation with the best model family
5. Alpha evaluation with Alphalens
6. Portfolio optimisation with Riskfolio-Lib
7. Backtest with VectorBT
8. Performance tear sheet with Pyfolio

> **Note:** This template uses synthetic data so it works fully offline.
> Swap the data source to `yfinance` or `openbb` for real market data.

## 0. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path so src/ is importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

from src.utils.config import load_config
from src.utils.logging import get_logger

logger = get_logger(__name__)
config = load_config()

TICKERS = config["universe"]["tickers"]
SEED = config["general"]["random_seed"]
print(f"Universe: {TICKERS}")
print(f"Seed: {SEED}")

---
## 1. Load Data

We generate synthetic OHLCV data for the UK large-cap universe using
geometric Brownian motion.  In production you would load processed Parquet
files from `data/processed/` instead (see the commented-out cell below).

In [ ]:
from src.data.synthetic import generate_multi_asset_data

# Generate ~5 years of correlated synthetic data
multi_data = generate_multi_asset_data(TICKERS, days=1260, seed=SEED)

print(f"Generated data for {len(multi_data)} tickers")
for ticker, df in multi_data.items():
    print(f"  {ticker}: {df.index.min().date()} → {df.index.max().date()}  ({len(df)} rows)")

# Quick peek at one ticker
multi_data[TICKERS[0]].tail(3)

In [ ]:
# --- ALTERNATIVE: Load from processed Parquet files ---
# Uncomment this block to use real data fetched via `python -m scripts.fetch_data`
#
# from src.utils.config import get_data_dir
# data_dir = get_data_dir(config) / "processed"
# multi_data = {}
# for ticker in TICKERS:
#     path = data_dir / f"{ticker}.parquet"
#     if path.exists():
#         multi_data[ticker] = pd.read_parquet(path)
#         print(f"Loaded {ticker}: {len(multi_data[ticker])} rows")
#     else:
#         print(f"WARNING: {path} not found — run scripts/fetch_data.py first")

### 1.1 Visualise Close Prices

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for ticker, df in multi_data.items():
    # Normalise to 100 at start for comparison
    normalised = df["Close"] / df["Close"].iloc[0] * 100
    ax.plot(normalised, label=ticker)

ax.set_title("Normalised Close Prices (base = 100)")
ax.set_ylabel("Price (rebased)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Feature Engineering

The `FeaturePipeline` applies technical indicators (SMA, EMA, RSI, MACD,
Bollinger Bands, ATR) and multi-horizon return features to each ticker.
Warm-up rows with NaN are dropped automatically.

In [ ]:
from src.features.pipeline import FeaturePipeline

pipeline = FeaturePipeline()
features_dict = pipeline.run(multi_data)

feature_names = pipeline.get_feature_names()
print(f"Generated {len(feature_names)} features: {feature_names[:10]}...")

# Show one ticker's features
sample_ticker = TICKERS[0]
features_dict[sample_ticker].head(3)

In [ ]:
# Correlation heatmap for the first ticker's features
feat_df = features_dict[sample_ticker][feature_names]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feat_df.corr(), cmap="RdBu_r", center=0, ax=ax, fmt=".1f",
            annot=True, annot_kws={"size": 7})
ax.set_title(f"Feature Correlation — {sample_ticker}")
plt.tight_layout()
plt.show()

### 2.1 Construct the Target Variable

We use a simple binary classification target: **1** if next-day return > 0,
**0** otherwise.  The target is shifted forward by one day so features use
only information available at time *t* to predict the return at *t+1*.

In [ ]:
# Build features and target for modelling (single ticker for now)
feat = features_dict[sample_ticker].copy()

# Target: next-day return direction (1 = up, 0 = down)
feat["target"] = (feat["Close"].pct_change().shift(-1) > 0).astype(int)
feat = feat.dropna(subset=["target"])

X = feat[feature_names]
y = feat["target"]

print(f"X shape: {X.shape}")
print(f"Target balance: {y.value_counts().to_dict()}")
print(f"Date range: {X.index.min().date()} → {X.index.max().date()}")

---
## 3. Quick ML Model Comparison (AutoML Screening)

Use PyCaret to rapidly compare model families.  This tells us whether
*any* algorithm can find predictive signal in the feature set before we
invest time in full walk-forward validation.

> **Requires PyCaret:** `pip install 'quant-stack[ml-extended]'`.
> If not installed, this cell falls back to a manual scikit-learn comparison.

In [ ]:
try:
    from src.models.automl import quick_compare

    leaderboard, best_automl_model = quick_compare(
        X, y,
        target_type="classification",
        sort="F1",
        session_id=SEED,
    )
    print("PyCaret leaderboard:")
    display(leaderboard)
    AUTOML_AVAILABLE = True

except ImportError:
    print("PyCaret not installed — falling back to manual model comparison.")
    AUTOML_AVAILABLE = False

    # Manual comparison using our classical models
    from src.models.classical import RandomForestModel, GradientBoostingModel
    from src.models.evaluation import walk_forward_cv

    models_to_compare = {
        "RandomForest": RandomForestModel(),
        "GradientBoosting": GradientBoostingModel(),
    }

    comparison_results = {}
    for name, model in models_to_compare.items():
        print(f"\nRunning walk-forward CV for {name}...")
        result = walk_forward_cv(model, X, y, n_splits=3, min_train_size=200)
        comparison_results[name] = result["mean_metrics"]
        print(f"  Mean metrics: {result['mean_metrics']}")

    leaderboard = pd.DataFrame(comparison_results).T
    print("\nModel comparison:")
    display(leaderboard)

---
## 4. Walk-Forward Cross-Validation

Proper time-series-aware evaluation of the best model family using an
expanding-window walk-forward scheme.  A gap of 5 trading days is inserted
between train and test periods to prevent autocorrelation leakage.

```
|---- train (expanding) ----|-- gap --|---- test fold k ----|
```

In [ ]:
from src.models.classical import RandomForestModel
from src.models.evaluation import walk_forward_cv

# Use RandomForest as the production model (swap based on automl results)
model = RandomForestModel()

cv_results = walk_forward_cv(
    model, X, y,
    n_splits=5,
    gap=5,
    min_train_size=252,
)

print("\nPer-fold metrics:")
folds_df = pd.DataFrame(cv_results["fold_metrics"])
folds_df.index.name = "fold"
display(folds_df)

print("\nMean metrics across folds:")
for k, v in cv_results["mean_metrics"].items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Visualise fold boundaries and per-fold Sharpe
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={"height_ratios": [2, 1]})

# Top: price with fold boundaries
close = feat["Close"]
ax1.plot(close, color="steelblue", linewidth=0.8)
for i, (train_end, test_start, test_end) in enumerate(cv_results["splits"]):
    ax1.axvspan(pd.Timestamp(test_start), pd.Timestamp(test_end),
                alpha=0.15, color=f"C{i}", label=f"Fold {i}")
ax1.set_title(f"Walk-Forward CV Splits — {sample_ticker}")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Bottom: per-fold metrics
metrics_to_plot = [c for c in folds_df.columns if c in ("accuracy", "sharpe")]
folds_df[metrics_to_plot].plot(kind="bar", ax=ax2)
ax2.set_title("Per-Fold Metrics")
ax2.set_ylabel("Value")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Alpha Evaluation with Alphalens

Evaluate the model's predictive signal as an alpha factor across the full
universe.  Alphalens computes information coefficients, factor returns,
and quantile turnover to assess signal quality.

In [ ]:
# Train a final model on 80% of data, predict on the remaining 20%
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

final_model = RandomForestModel()
final_model.fit(X_train, y_train)

# Generate CONTINUOUS probability scores for ALL tickers in the test period.
# Binary predictions (0/1) are unsuitable for Alphalens quantile analysis —
# we need a continuous factor so that assets can be ranked meaningfully.
test_dates = X_test.index
factor_rows = []

for ticker in TICKERS:
    ticker_feat = features_dict[ticker][feature_names]
    ticker_test = ticker_feat.loc[ticker_feat.index.isin(test_dates)]
    if ticker_test.empty:
        continue
    # Use predict_proba for a continuous signal (probability of up-day)
    proba = final_model._estimator.predict_proba(ticker_test.values)[:, 1]
    for date, score in zip(ticker_test.index, proba):
        factor_rows.append((date, ticker, float(score)))

factor_series = pd.Series(
    {(date, ticker): val for date, ticker, val in factor_rows},
    name="factor",
)
factor_series.index = pd.MultiIndex.from_tuples(
    factor_series.index, names=["date", "asset"]
)

# Build a close-price DataFrame for Alphalens
closes = pd.DataFrame({t: multi_data[t]["Close"] for t in TICKERS})

print(f"Factor observations: {len(factor_series)}")
print(f"Price matrix: {closes.shape}")
print(f"Factor range: {factor_series.min():.3f} – {factor_series.max():.3f}")

In [ ]:
from src.portfolio.analysis import evaluate_alpha

# With only 5 assets, quantile binning can be fragile. Using bins
# (equal-width intervals on the probability score) is more robust
# for small universes. Switch to quantiles=5 with larger universes.
try:
    alpha_report = evaluate_alpha(
        factor_series,
        closes,
        periods=(1, 5, 21),
        quantiles=3,
        max_loss=0.5,
    )
    ALPHA_OK = True
except Exception as e:
    print(f"Alphalens quantile binning failed ({e}).")
    print("Retrying with equal-width bins instead of quantiles...")
    import alphalens.utils as al_utils
    import alphalens.performance as al_perf

    factor_data = al_utils.get_clean_factor_and_forward_returns(
        factor_series, closes,
        bins=3,
        periods=(1, 5, 21),
        max_loss=1.0,
    )
    ic_by_period = al_perf.factor_information_coefficient(factor_data)
    ic_means = al_perf.mean_information_coefficient(factor_data)
    factor_returns = al_perf.factor_returns(factor_data)
    alpha_report = {
        "factor_data": factor_data,
        "ic": ic_means,
        "ic_by_period": ic_by_period,
        "factor_returns": factor_returns,
        "summary": {},
    }
    for p in (1, 5, 21):
        col = f"{p}D"
        mean_ic = float(ic_means.loc[col]) if col in ic_means.index else 0.0
        alpha_report["summary"][col] = {
            "mean_ic": round(mean_ic, 4),
            "annualised_factor_return": round(float(factor_returns[col].mean()) * 252, 6),
            "top_quantile_turnover": 0.0,
            "signal_quality": "strong" if abs(mean_ic) >= 0.05 else (
                "moderate" if abs(mean_ic) >= 0.02 else "weak"),
        }
    ALPHA_OK = True

if ALPHA_OK:
    print("\nAlpha evaluation summary:")
    for period, metrics in alpha_report["summary"].items():
        print(f"  {period}: IC={metrics['mean_ic']:.4f}, "
              f"Ann. factor ret={metrics['annualised_factor_return']:.6f}, "
              f"Quality={metrics['signal_quality']}")

In [ ]:
# Plot Information Coefficient over time
ic_df = alpha_report["ic_by_period"]
fig, axes = plt.subplots(1, ic_df.shape[1], figsize=(14, 4), sharey=True)
if ic_df.shape[1] == 1:
    axes = [axes]

for ax, col in zip(axes, ic_df.columns):
    ic_col = ic_df[col].dropna()
    ax.bar(ic_col.index, ic_col.values, width=2, alpha=0.6)
    ax.axhline(y=ic_col.mean(), color="red", linestyle="--",
               label=f"Mean IC = {ic_col.mean():.4f}")
    ax.set_title(f"IC — {col}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Information Coefficient Over Time", y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Portfolio Optimisation with Riskfolio-Lib

Given the model's alpha signals, optimise portfolio weights using several
methods: mean-variance, minimum CVaR, risk parity, and equal weight.
Position-size constraints from `config/settings.yaml` are enforced
automatically.

In [ ]:
from src.portfolio.optimiser import PortfolioOptimiser

# Build returns matrix from close prices
returns = closes.pct_change().dropna()
print(f"Returns matrix: {returns.shape}")
print(f"Date range: {returns.index.min().date()} → {returns.index.max().date()}")

In [ ]:
optimiser = PortfolioOptimiser(config=config)

methods = ["equal_weight", "mean_variance", "min_cvar", "risk_parity"]
all_weights = {}

for method in methods:
    try:
        w = optimiser.optimise(returns, method=method)
        all_weights[method] = w
        print(f"\n{method}:")
        for ticker, weight in w.items():
            print(f"  {ticker}: {weight:.4f}")
    except Exception as e:
        print(f"\n{method}: FAILED — {e}")

In [ ]:
# Compare weight allocations across methods
weights_df = pd.DataFrame(all_weights)

fig, ax = plt.subplots(figsize=(12, 5))
weights_df.plot(kind="bar", ax=ax)
ax.set_title("Portfolio Weights by Optimisation Method")
ax.set_ylabel("Weight")
ax.set_xlabel("Ticker")
ax.axhline(y=1.0 / len(TICKERS), color="grey", linestyle="--", alpha=0.5,
           label=f"Equal = {1.0/len(TICKERS):.2f}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 6.1 Risk Metrics

Compute VaR, CVaR, and correlation matrix for the optimised portfolio.

In [ ]:
from src.portfolio.risk import portfolio_var, portfolio_cvar, correlation_matrix

# Use mean-variance weights (or fall back to equal weight)
chosen_weights = all_weights.get("mean_variance", all_weights["equal_weight"])
chosen_method = "mean_variance" if "mean_variance" in all_weights else "equal_weight"

var_95 = portfolio_var(returns, chosen_weights, confidence=0.95)
cvar_95 = portfolio_cvar(returns, chosen_weights, confidence=0.95)

print(f"Portfolio risk metrics ({chosen_method} weights):")
print(f"  95% VaR  (daily): {var_95:.4f} ({var_95*100:.2f}%)")
print(f"  95% CVaR (daily): {cvar_95:.4f} ({cvar_95*100:.2f}%)")

# Correlation matrix with high-correlation warnings
corr = correlation_matrix(returns, flag_threshold=0.85)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Return Correlation Matrix")
plt.tight_layout()
plt.show()

---
## 7. Backtest with VectorBT

Run a full backtest using the built-in momentum and mean-reversion
strategies with realistic transaction costs (commission + slippage from
config).  We test on a single ticker here; extend to the full universe
for portfolio-level backtesting.

In [ ]:
from src.backtest.engine import run_backtest, compare_strategies
from src.backtest.strategy import MomentumStrategy, MeanReversionStrategy

# Use the first ticker's feature DataFrame
bt_data = features_dict[sample_ticker]

strategies = [
    MomentumStrategy.from_config(config),
    MeanReversionStrategy.from_config(config),
]

# Compare strategies side by side
comparison = compare_strategies(strategies, bt_data, config)
print("Strategy comparison:")
display(comparison)

In [ ]:
# Run the best strategy individually for detailed analysis
best_strat_name = comparison["sharpe_ratio"].idxmax()
best_strat = [s for s in strategies if s.name == best_strat_name][0]
result = run_backtest(best_strat, bt_data, config)

print(f"\nBest strategy: {result.strategy_name}")
print(f"  Total return: {result.metrics['total_return']:.2%}")
print(f"  Annualised return: {result.metrics['annualised_return']:.2%}")
print(f"  Sharpe ratio: {result.metrics['sharpe_ratio']:.2f}")
print(f"  Max drawdown: {result.metrics['max_drawdown']:.2%}")
print(f"  Calmar ratio: {result.metrics['calmar_ratio']:.2f}")
print(f"  Total trades: {result.metrics['total_trades']:.0f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [2, 1, 1]})

# Equity curve
ax = axes[0]
ax.plot(result.equity_curve, label="Strategy equity", linewidth=1)
ax.axhline(y=result.metrics["initial_capital"], color="grey", linestyle="--", alpha=0.5)
ax.set_title(f"Equity Curve — {result.strategy_name}")
ax.set_ylabel("Portfolio Value (GBP)")
ax.legend()
ax.grid(True, alpha=0.3)

# Drawdown
ax = axes[1]
cum_max = result.equity_curve.cummax()
drawdown = (result.equity_curve - cum_max) / cum_max
ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.5, color="red")
ax.set_title("Drawdown")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.3)

# Signals
ax = axes[2]
ax.plot(result.signals, linewidth=0.5, alpha=0.8)
ax.set_title("Trading Signals")
ax.set_ylabel("Signal")
ax.set_yticks([-1, 0, 1])
ax.set_yticklabels(["Short", "Flat", "Long"])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Performance Tear Sheet (Pyfolio)

Generate a comprehensive performance report: annualised return, Sharpe,
Sortino, Calmar, max drawdown, and (if a benchmark is available)
excess return and information ratio.

In [ ]:
from src.portfolio.analysis import generate_tearsheet

tearsheet = generate_tearsheet(
    result.returns,
    risk_free_rate=config["portfolio"]["risk_free_rate"],
)

print("Performance metrics:")
for metric, value in tearsheet["metrics"].items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value}")

In [ ]:
from src.portfolio.risk import rolling_sharpe

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Cumulative returns
ax = axes[0]
cum_ret = (1 + result.returns).cumprod()
ax.plot(cum_ret, label="Strategy", linewidth=1)
ax.axhline(y=1.0, color="grey", linestyle="--", alpha=0.5)
ax.set_title("Cumulative Returns")
ax.set_ylabel("Growth of \u00a31")
ax.legend()
ax.grid(True, alpha=0.3)

# Rolling Sharpe
ax = axes[1]
rs = rolling_sharpe(result.returns, window=126, risk_free_rate=0.045)
ax.plot(rs, linewidth=0.8)
ax.axhline(y=0, color="grey", linestyle="--", alpha=0.5)
ax.set_title("Rolling Sharpe Ratio (6-month window)")
ax.set_ylabel("Sharpe")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

This notebook walked through the full Quant Stack research workflow:

| Stage | Module | Output |
|-------|--------|--------|
| 1. Data | `src.data.synthetic` / `fetcher` | OHLCV DataFrames |
| 2. Features | `src.features.pipeline` | Technical indicators + returns |
| 3. AutoML | `src.models.automl` | Model family screening |
| 4. Validation | `src.models.evaluation` | Walk-forward CV metrics |
| 5. Alpha | `src.portfolio.analysis` | IC, factor returns, turnover |
| 6. Portfolio | `src.portfolio.optimiser` | Optimised weights |
| 7. Backtest | `src.backtest.engine` | Equity curve, trades, metrics |
| 8. Tear sheet | `src.portfolio.analysis` | Sharpe, Sortino, drawdown |

### Next steps
- Swap synthetic data for real market data (`python -m scripts.fetch_data`)
- Experiment with new features in `src/features/`
- Add custom strategies in `src/backtest/strategy.py`
- Deploy to paper trading via `python -m scripts.run_rebalance`